In [1]:
# Essential imports for JSON reading
import json
import os
import sys
from typing import List, Dict, Any

print("✅ Essential imports loaded successfully!")

from typing import List, Dict, Any, Optional
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)



# Importing Required Libraries

# Fix numpy compatibility issue with gensim
import warnings
warnings.filterwarnings('ignore', message='.*numpy.dtype size changed.*')
warnings.filterwarnings('ignore', message='.*binary incompatibility.*')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import PCA
import os
import csv
import re
from dotenv import load_dotenv
import json 

# Fix VoyageAI API Key Loading
import voyageai
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/code/conf/project_details.env')

# Get API key from environment
voyage_api_key = os.getenv('VOYAGE_API_KEY')
print(f"API Key loaded: {voyage_api_key[:10]}..." if voyage_api_key else "No API key found!")




# Try to import with error handling
try:
    from contor_utils_5 import _read_dictionary
    print("✅ Utils import successful!")
except ImportError as e:
    print(f"❌ Utils import error: {e}")




# Handle gensim import separately to avoid numpy compatibility issues
try:
    # Try to suppress the specific numpy compatibility warning
    import os
    os.environ['PYTHONWARNINGS'] = 'ignore::UserWarning'
    
    from gensim.models import KeyedVectors
    GENSIM_AVAILABLE = True
    print("✅ Gensim import successful!")
except Exception as e:
    print(f"❌ Gensim import error: {e}")
    print("Word embeddings will use fallback mode (zero embeddings).")
    GENSIM_AVAILABLE = False

✅ Essential imports loaded successfully!


INFO:numexpr.utils:NumExpr defaulting to 12 threads.


API Key loaded: pa-ZnppvtJ...
DGL functionality will be limited
✅ Utils import successful!
✅ Gensim import successful!


Preprocess Contor Dataset


🚀 Voyage AI Embedding Generation for CONTOR Dataset
📦 Loaded 384 cached embeddings from voyage_embeddings_cache.json

📊 Found 697 unique concepts across all splits
📝 Getting embeddings for 313 concepts (384 from cache)
🔄 Processing batch 1/3 (128 concepts)...
   ✅ Batch 1 completed
🔄 Processing batch 2/3 (128 concepts)...
   ✅ Batch 2 completed
🔄 Processing batch 3/3 (57 concepts)...
   ✅ Batch 3 completed
💾 Saved embeddings cache to voyage_embeddings_cache.json

✅ Embedding generation complete!
   📊 Total concepts: 697
   📐 Embedding dimension: 1024
   🤖 Model: voyage-3-large
   📦 Embedding matrix shape: (697, 1024)


In [5]:


def preprocess_contor_data_new(json_file_path, 
                          ftype='voyage', 
                          dim=1024,
                          voyage_embeddings=None,
                          voyage_client=None,
                          voyage_model='voyage-3-large',
                          voyage_input_type='document',
                          bidirectional=False,
                          cache_dir=None,
                          return_dict=True,
                          verbose=True):
    """
    Enhanced preprocess CONTOR train.json with Voyage embeddings as default.
    All functionalities from load_whole_data, optimized for Voyage AI embeddings.
    
    Args:
        json_file_path: Path to CONTOR JSON file (one JSON object per line)
        ftype: Feature type - 'voyage' (default), 'analogy', 'embedding', or 'none'
        dim: Dimension for embeddings (default: 1024 for Voyage-3-large)
        voyage_embeddings: Pre-computed Voyage embeddings dict {concept: embedding_array}
                          If None and ftype='voyage', will generate using voyage_client
        voyage_client: VoyageAI client instance (if None, will try to create from env)
        voyage_model: Voyage model name (default: 'voyage-3-large')
        voyage_input_type: Input type for Voyage ('document' or 'query', default: 'document')
        bidirectional: If True, add reverse edges (bidirectional)
        cache_dir: Directory to cache nodes_dict, label_dict, and features files
        return_dict: If True, return dictionary; if False, return tuple like load_whole_data
        verbose: Whether to print progress information
        
    Returns:
        If return_dict=True:
            Dictionary with all components
        If return_dict=False:
            Tuple: (num_node, edge_list, edge_src, edge_dst, edge_type, edge_norm, 
                   num_rel, node_id_con, labels, node_features)
    """
    
    # 1. Load CONTOR JSON data
    if verbose:
        print(f"📖 Loading CONTOR data from: {json_file_path}")
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    
    if verbose:
        print(f"✅ Loaded {len(data)} samples")
    
    # Determine cache file paths
    if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)
        node_dict_file = os.path.join(cache_dir, 'all_nodes.dict')
        label_dict_file = os.path.join(cache_dir, 'all_unary_templates.dict')
    else:
        node_dict_file = None
        label_dict_file = None
    
    # 2. Create nodes dictionary with caching
    nodes_dict = None
    if node_dict_file and os.path.exists(node_dict_file):
        if verbose:
            print(f"📦 Loading nodes_dict from cache: {node_dict_file}")
        try:
            nodes_dict = _read_dictionary(node_dict_file)
        except Exception as e:
            if verbose:
                print(f"⚠️  Warning: Could not load cache: {e}")
            nodes_dict = None
    
    if nodes_dict is None:
        if verbose:
            print("📝 Creating nodes_dict from data...")
        nodes_dict = {}
        all_concepts = set()
        
        for item in data:
            sub_concept = item['v_sub_concept']
            super_concept = item['v_super_concept']
            all_concepts.add(sub_concept)
            all_concepts.add(super_concept)
        
        # Assign node IDs
        for i, concept in enumerate(sorted(all_concepts)):
            nodes_dict[concept] = i
        
        # Save to cache
        if node_dict_file:
            if verbose:
                print(f"💾 Saving nodes_dict to cache: {node_dict_file}")
            node_id_str = ''
            for nod_id, nod in enumerate(sorted(all_concepts)):
                node_id_str += str(nod_id) + '\t' + nod + '\n'
            with open(node_dict_file, 'w', encoding='utf-8') as f:
                f.write(node_id_str)
    
    num_nodes = len(nodes_dict)
    if verbose:
        print(f"📊 Number of nodes: {num_nodes}")
    
    # 3. Create label dictionary with caching
    label_dict = None
    if label_dict_file and os.path.exists(label_dict_file):
        if verbose:
            print(f"📦 Loading label_dict from cache: {label_dict_file}")
        try:
            label_dict = _read_dictionary(label_dict_file)
        except Exception as e:
            if verbose:
                print(f"⚠️  Warning: Could not load cache: {e}")
            label_dict = None
    
    if label_dict is None:
        if verbose:
            print("📝 Creating label_dict from data...")
        label_dict = {}
        label_id = 0
        
        for item in data:
            rule = item['rule']
            if rule not in label_dict:
                label_dict[rule] = label_id
                label_id += 1
        
        # Save to cache
        if label_dict_file:
            if verbose:
                print(f"💾 Saving label_dict to cache: {label_dict_file}")
            label_id_str = ''
            for lab_id, lab in enumerate(sorted(label_dict.keys(), key=lambda x: label_dict[x])):
                label_id_str += str(lab_id) + '\t' + lab + '\n'
            with open(label_dict_file, 'w', encoding='utf-8') as f:
                f.write(label_id_str)
    
    num_labels = len(label_dict)
    if verbose:
        print(f"📊 Number of labels: {num_labels}")
    
    # 4. Create label matrix
    if verbose:
        print("📝 Creating label matrix...")
    labels = sp.lil_matrix((num_nodes, num_labels))
    
    for item in data:
        sub_concept = item['v_sub_concept']
        rule = item['rule']
        label = item['label']
        
        if sub_concept in nodes_dict and rule in label_dict:
            node_id = nodes_dict[sub_concept]
            label_id = label_dict[rule]
            labels[node_id, label_id] = label
    
    labels = labels.tocsr()
    
    # 5. Create edge list (concept relationships)
    if verbose:
        print("📝 Creating edge list...")
    edge_list = []
    relation_dict = {}
    rid = 1  # 0 for self-relation
    node_id_con = set()  # nodes connected with others
    
    for item in data:
        sub_concept = item['v_sub_concept']
        super_concept = item['v_super_concept']
        label = item['label']
        
        if sub_concept in nodes_dict and super_concept in nodes_dict:
            # Create relationship edge
            relation_type = f"IS_A_{label}"  # Different relation types for valid/invalid
            if relation_type not in relation_dict:
                relation_dict[relation_type] = rid
                rid += 1
            
            src = nodes_dict[sub_concept]
            dst = nodes_dict[super_concept]
            node_id_con.add(src)
            node_id_con.add(dst)
            
            edge_list.append((src, dst, relation_dict[relation_type]))
            
            # Add bidirectional edge if requested
            if bidirectional:
                edge_list.append((dst, src, relation_dict[relation_type]))
    
    # 6. Add self-connections (only for connected nodes, like load_whole_data)
    for node_id in node_id_con:
        edge_list.append((node_id, node_id, 0))  # Self-relation
    
    # 7. Sort edges (like load_whole_data)
    edge_list = sorted(edge_list, key=lambda x: (x[1], x[0], x[2]))
    edge_list = np.array(edge_list, dtype=np.int32)
    
    num_rel = len(relation_dict) + 1  # +1 for self-relation
    
    # 8. Compute edge normalization (like load_whole_data)
    if verbose:
        print("📝 Computing edge normalization...")
    edge_src, edge_dst, edge_type = edge_list.transpose()
    _, inverse_index, count = np.unique((edge_dst, edge_type), axis=1, 
                                        return_inverse=True, return_counts=True)
    degrees = count[inverse_index]  # c_{i,r} for each relation type
    edge_norm = np.ones(len(edge_dst), dtype=np.float32) / degrees.astype(np.float32)
    
    # 9. Generate node features (Voyage embeddings as default)
    node_features = None
    if ftype != 'none':
        if cache_dir:
            if ftype == 'voyage':
                feature_file = os.path.join(cache_dir, f'all_voyage_features_{dim}.csv')
            elif ftype == 'analogy':
                feature_file = os.path.join(cache_dir, f'all_an_features_{dim}.csv')
            elif ftype == 'embedding':
                feature_file = os.path.join(cache_dir, 'all_em_features.csv')
            else:
                feature_file = None
        else:
            feature_file = None
        
        if feature_file and os.path.exists(feature_file):
            if verbose:
                print(f"📦 Loading node features from cache: {feature_file}")
            try:
                node_features = pd.read_csv(feature_file, sep=',', encoding='utf-8', header=None)
                node_features = node_features.values
                if verbose:
                    print(f"✅ Loaded features with shape: {node_features.shape}")
            except Exception as e:
                if verbose:
                    print(f"⚠️  Warning: Could not load cached features: {e}")
                node_features = None
        
        if node_features is None:
            if verbose:
                print(f"📝 Generating node features (type: {ftype})...")
            
            if ftype == 'voyage':
                # Voyage embeddings (primary/default method)
                if voyage_embeddings:
                    # Use provided Voyage embeddings
                    if verbose:
                        print(f"📦 Using provided Voyage embeddings...")
                    node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                    missing_count = 0
                    for concept, node_id in nodes_dict.items():
                        if concept in voyage_embeddings:
                            emb = voyage_embeddings[concept]
                            # Handle both array and list formats
                            if isinstance(emb, (list, tuple)):
                                emb = np.array(emb)
                            # Truncate or pad to desired dimension
                            if len(emb) >= dim:
                                node_features[node_id] = emb[:dim]
                            else:
                                # Pad with zeros if embedding is shorter
                                node_features[node_id, :len(emb)] = emb
                        else:
                            missing_count += 1
                            if verbose and missing_count <= 5:
                                print(f"⚠️  Warning: No Voyage embedding for concept '{concept}'")
                    if verbose and missing_count > 5:
                        print(f"⚠️  Warning: {missing_count} concepts missing Voyage embeddings")
                
                elif voyage_client:
                    # Generate Voyage embeddings on the fly
                    if verbose:
                        print(f"🚀 Generating Voyage embeddings using {voyage_model}...")
                    try:
                        import voyageai
                        concepts_list = sorted(list(nodes_dict.keys()))
                        texts_to_embed = [f"Transportation concept: {concept}" for concept in concepts_list]
                        
                        # Batch processing
                        batch_size = 128
                        all_embeddings_list = []
                        
                        for i in range(0, len(texts_to_embed), batch_size):
                            batch_texts = texts_to_embed[i:i + batch_size]
                            if verbose:
                                print(f"   Processing batch {i//batch_size + 1}/{(len(texts_to_embed)-1)//batch_size + 1}...")
                            
                            result = voyage_client.embed(
                                batch_texts,
                                model=voyage_model,
                                input_type=voyage_input_type,
                                truncation=True
                            )
                            all_embeddings_list.extend(result.embeddings)
                        
                        # Create feature matrix
                        node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                        for idx, concept in enumerate(concepts_list):
                            emb = np.array(all_embeddings_list[idx])
                            if len(emb) >= dim:
                                node_features[nodes_dict[concept]] = emb[:dim]
                            else:
                                node_features[nodes_dict[concept], :len(emb)] = emb
                        
                        if verbose:
                            print(f"✅ Generated Voyage embeddings with shape: {node_features.shape}")
                    
                    except Exception as e:
                        if verbose:
                            print(f"❌ Error generating Voyage embeddings: {e}")
                            print("⚠️  Falling back to zero embeddings")
                        node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                
                else:
                    # Try to get from environment or create client
                    try:
                        import voyageai
                        voyage_api_key = os.getenv('VOYAGE_API_KEY')
                        if voyage_api_key:
                            if verbose:
                                print(f"🔑 Creating Voyage client from API key...")
                            voyage_client = voyageai.Client(api_key=voyage_api_key)
                            # Recursive call with client
                            # Extract relevant parameters and call again
                            return preprocess_contor_data_new(
                                json_file_path, ftype='voyage', dim=dim,
                                voyage_client=voyage_client, voyage_model=voyage_model,
                                voyage_input_type=voyage_input_type, bidirectional=bidirectional,
                                cache_dir=cache_dir, return_dict=return_dict, verbose=verbose
                            )
                        else:
                            if verbose:
                                print("⚠️  Warning: No Voyage API key found and no embeddings provided")
                            node_features = np.zeros((num_nodes, dim), dtype=np.float32)
                    except ImportError:
                        if verbose:
                            print("⚠️  Warning: voyageai package not available")
                        node_features = np.zeros((num_nodes, dim), dtype=np.float32)
            
            elif ftype == 'analogy':
                # Use PCA on labels
                labels_dense = labels.todense()
                pca = PCA(n_components=dim)
                node_features = pca.fit_transform(labels_dense)
                if verbose:
                    print(f"✅ Generated analogy features with shape: {node_features.shape}")
            
            elif ftype == 'embedding':
                # Fallback to Word2Vec (if needed)
                if verbose:
                    print("⚠️  Note: Using Word2Vec embeddings (consider using Voyage instead)")
                try:
                    from word_embedding_2 import word_embedding
                    embedding_file = 'dataset/GoogleNews-vectors-negative300.bin.gz'
                    node_features = word_embedding(embedding_file, nodes_dict)
                except ImportError:
                    if verbose:
                        print("⚠️  Warning: word_embedding function not available, using zero embeddings")
                    node_features = np.zeros((num_nodes, dim), dtype=np.float32)
            
            # Save features to cache
            if feature_file and node_features is not None:
                if verbose:
                    print(f"💾 Saving node features to cache: {feature_file}")
                try:
                    with open(feature_file, 'w', encoding='utf-8', newline='') as f:
                        writer = csv.writer(f)
                        # Write without header row for consistency
                        writer.writerows(node_features)
                    if verbose:
                        print(f"✅ Saved features to {feature_file}")
                except Exception as e:
                    if verbose:
                        print(f"⚠️  Warning: Could not save features to cache: {e}")
    
    # Convert labels to dense format (like load_whole_data)
    labels = labels.todense()
    
    if verbose:
        print("✅ Preprocessing complete!")
        print(f"   Nodes: {num_nodes}, Edges: {len(edge_list)}, Relations: {num_rel}")
        if node_features is not None:
            print(f"   Features shape: {node_features.shape}")
    
    # Return format
    if return_dict:
        return {
            'nodes_dict': nodes_dict,
            'label_dict': label_dict,
            'train_labels': labels,
            'labels': labels,
            'edge_list': edge_list,
            'relation_dict': relation_dict,
            'edge_src': edge_src,
            'edge_dst': edge_dst,
            'edge_type': edge_type,
            'edge_norm': edge_norm,
            'num_node': num_nodes,
            'num_rel': num_rel,
            'node_id_con': node_id_con,
            'node_features': node_features
        }
    else:
        # Return tuple format like load_whole_data
        return (num_nodes, edge_list, edge_src, edge_dst, edge_type, edge_norm, 
                num_rel, node_id_con, labels, node_features)

In [6]:


# Custom model and dimensions
result = preprocess_contor_data_new("/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json", voyage_model='voyage-3', dim=1024)

📖 Loading CONTOR data from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
✅ Loaded 1904 samples
📝 Creating nodes_dict from data...
📊 Number of nodes: 618
📝 Creating label_dict from data...
📊 Number of labels: 1904
📝 Creating label matrix...
📝 Creating edge list...
📝 Computing edge normalization...
📝 Generating node features (type: voyage)...
🔑 Creating Voyage client from API key...
📖 Loading CONTOR data from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
✅ Loaded 1904 samples
📝 Creating nodes_dict from data...
📊 Number of nodes: 618
📝 Creating label_dict from data...
📊 Number of labels: 1904
📝 Creating label matrix...
📝 Creating edge list...
📝 Computing edge normalization...
📝 Generating node features (type: voyage)...
🚀 Generating Voyage embeddings using voyage-3...
   Processing batch 1/5...
   Processing batch 2/5...
   Processing batch 3/

In [9]:
result.keys()

dict_keys(['nodes_dict', 'label_dict', 'train_labels', 'labels', 'edge_list', 'relation_dict', 'edge_src', 'edge_dst', 'edge_type', 'edge_norm', 'num_node', 'num_rel', 'node_id_con', 'node_features'])

In [10]:
result['nodes_dict']

{'agent': 0,
 'aid to navigation': 1,
 'air route': 2,
 'air traffic control center': 3,
 'air traffic control en route center': 4,
 'air traffic control procedure': 5,
 'air traffic control radar room': 6,
 'air transitway': 7,
 'aircraft': 8,
 'airplane': 9,
 'airport': 10,
 'airport by runway surface': 11,
 'airport classification': 12,
 'ambulance': 13,
 'anchorage': 14,
 'animal powered device': 15,
 'animal powered vehicle': 16,
 'arriving': 17,
 'artifact': 18,
 'atmospheric region': 19,
 'attaching': 20,
 'automobile': 21,
 'axle': 22,
 'barge': 23,
 'barge carrier ship': 24,
 'battery': 25,
 'bicycle': 26,
 'bill of lading': 27,
 'body motion': 28,
 'boxcar': 29,
 'bridge': 30,
 'broad gauge railway': 31,
 'building': 32,
 'bulk cargo': 33,
 'bulkhead': 34,
 'bus': 35,
 'business railcar': 36,
 'c i a airport length classification': 37,
 'cab car': 38,
 'cable ship': 39,
 'cabotage': 40,
 'canal': 41,
 'canal lock': 42,
 'canal lock gate': 43,
 'canal structure': 44,
 'canoe':

In [11]:
result['label_dict']

{'body=Mid-level-ontology.Tunnel, head=SUMO.StationaryArtifact': 0,
 'body=Mid-level-ontology.Pipeline, Mid-level-ontology.Tunnel, head=owl.Bottom': 1,
 'body=transport.RailwayJunction, head=Mid-level-ontology.TransitwayJunction': 2,
 'body=Mid-level-ontology.Tunnel, Mid-level-ontology.Waterway, head=owl.Bottom': 3,
 'body=transport.MerchantMarine, head=SUMO.Collection': 4,
 'body=transport.NavigationLight, head=transport.AidToNavigation': 5,
 'body=transport.Barge, head=Mid-level-ontology.Watercraft': 6,
 'body=Mid-level-ontology.Bridge, head=SUMO.LandTransitway': 7,
 'body=Mid-level-ontology.Railway, head=SUMO.LandTransitway': 8,
 'body=Mid-level-ontology.AnimalPoweredDevice, transport.TransportationControlDevice, head=owl.Bottom': 9,
 'body=Mid-level-ontology.Waterway, head=SUMO.WaterArea': 10,
 'body=transport.Locomotive, head=transport.RollingStock': 11,
 'body=transport.ChemicalTankerShip, head=transport.CargoShip': 12,
 'body=transport.RailroadBridge, head=Mid-level-ontology.Rai

In [12]:
result['train_labels']

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

In [13]:
result['labels']

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
result['edge_list']

array([[  0,   0,   0],
       [125,   0,   2],
       [153,   0,   2],
       ...,
       [616, 616,   0],
       [107, 617,   2],
       [617, 617,   0]], dtype=int32)

In [15]:
result['relation_dict']

{'IS_A_1': 1, 'IS_A_0': 2}

In [16]:
result['edge_src']

array([  0, 125, 153, ..., 616, 107, 617], dtype=int32)

In [17]:
result['edge_dst']

array([  0,   0,   0, ..., 616, 617, 617], dtype=int32)

In [18]:
result['edge_type']

array([0, 2, 2, ..., 0, 2, 0], dtype=int32)

In [19]:
result['edge_norm']

array([1.   , 0.125, 0.125, ..., 1.   , 1.   , 1.   ], dtype=float32)

In [20]:
result['num_node']

618

In [21]:
result['node_id_con']

{0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [22]:
result['node_features']

array([[ 0.05305956, -0.02155073, -0.01985572, ..., -0.01568919,
         0.02364774,  0.04780785],
       [ 0.02048378, -0.06020527,  0.0143179 , ..., -0.02883693,
         0.00857864,  0.06401259],
       [ 0.03720454, -0.05270103, -0.00284776, ..., -0.03429271,
         0.04349302,  0.06440222],
       ...,
       [ 0.09509031, -0.04880908, -0.02238131, ..., -0.01647557,
         0.01436379,  0.06045818],
       [ 0.08823939, -0.05191094, -0.02483429, ..., -0.00427359,
         0.01690271,  0.05169683],
       [ 0.07342827, -0.075927  , -0.01500777, ..., -0.02427651,
        -0.00142639,  0.03597444]], dtype=float32)

In [4]:


json_file_path = "/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json"
ftype = 'voyage'
dim = 1024
voyage_embeddings = None
voyage_client = None
voyage_model = 'voyage-3-large'
voyage_input_type = 'document'
bidirectional = False
cache_dir = None
return_dict = True
verbose = True






In [14]:
# 1. Load CONTOR JSON data
if verbose:
    print(f"📖 Loading CONTOR data from: {json_file_path}")
with open(json_file_path, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

if verbose:
        print(f"✅ Loaded {len(data)} samples")

📖 Loading CONTOR data from: /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/contor_datasets/transport/train.json
✅ Loaded 1904 samples


In [15]:
if cache_dir:
        os.makedirs(cache_dir, exist_ok=True)
        node_dict_file = os.path.join(cache_dir, 'all_nodes.dict')
        label_dict_file = os.path.join(cache_dir, 'all_unary_templates.dict')

else:
        node_dict_file = None
        label_dict_file = None

In [22]:
# 2. Create nodes dictionary with caching
nodes_dict = None
if node_dict_file and os.path.exists(node_dict_file):
    if verbose:
        print(f"📦 Loading nodes_dict from cache: {node_dict_file}")
    try:
        nodes_dict = _read_dictionary(node_dict_file)
    except Exception as e:
        if verbose:
            print(f"⚠️  Warning: Could not load cache: {e}")
        nodes_dict = None

if nodes_dict is None:
        if verbose:
            print("📝 Creating nodes_dict from data...")
        nodes_dict = {}
        all_concepts = set()
        
        for item in data:
            sub_concept = item['v_sub_concept']
            super_concept = item['v_super_concept']
            all_concepts.add(sub_concept)
            all_concepts.add(super_concept)

# Assign node IDs
        for i, concept in enumerate(sorted(all_concepts)):
            nodes_dict[concept] = i


            # Save to cache
        if node_dict_file:
            if verbose:
                print(f"💾 Saving nodes_dict to cache: {node_dict_file}")
            node_id_str = ''
            for nod_id, nod in enumerate(sorted(all_concepts)):
                node_id_str += str(nod_id) + '\t' + nod + '\n'
            with open(node_dict_file, 'w', encoding='utf-8') as f:
                f.write(node_id_str)

num_nodes = len(nodes_dict)
if verbose:
        print(f"📊 Number of nodes: {num_nodes}")

📝 Creating nodes_dict from data...
📊 Number of nodes: 618


In [23]:
nodes_dict

{'agent': 0,
 'aid to navigation': 1,
 'air route': 2,
 'air traffic control center': 3,
 'air traffic control en route center': 4,
 'air traffic control procedure': 5,
 'air traffic control radar room': 6,
 'air transitway': 7,
 'aircraft': 8,
 'airplane': 9,
 'airport': 10,
 'airport by runway surface': 11,
 'airport classification': 12,
 'ambulance': 13,
 'anchorage': 14,
 'animal powered device': 15,
 'animal powered vehicle': 16,
 'arriving': 17,
 'artifact': 18,
 'atmospheric region': 19,
 'attaching': 20,
 'automobile': 21,
 'axle': 22,
 'barge': 23,
 'barge carrier ship': 24,
 'battery': 25,
 'bicycle': 26,
 'bill of lading': 27,
 'body motion': 28,
 'boxcar': 29,
 'bridge': 30,
 'broad gauge railway': 31,
 'building': 32,
 'bulk cargo': 33,
 'bulkhead': 34,
 'bus': 35,
 'business railcar': 36,
 'c i a airport length classification': 37,
 'cab car': 38,
 'cable ship': 39,
 'cabotage': 40,
 'canal': 41,
 'canal lock': 42,
 'canal lock gate': 43,
 'canal structure': 44,
 'canoe':

In [61]:
# 3. Create label dictionary with caching
label_dict = None
if label_dict_file and os.path.exists(label_dict_file):
    if verbose:
        print(f"📦 Loading label_dict from cache: {label_dict_file}")
    try:
        label_dict = _read_dictionary(label_dict_file)
    except Exception as e:
        if verbose:
            print(f"⚠️  Warning: Could not load cache: {e}")
        label_dict = None

if label_dict is None:
        if verbose:
            print("📝 Creating label_dict from data...")
        label_dict = {}
        label_id = 0
        

📝 Creating label_dict from data...


In [68]:
for item in data:

    if item['label'] == 1:

        for idx, template in enumerate(item['rule_template']):
            rule = template 
            print(rule)

            if rule not in label_dict:
                label_dict[rule] = label_id 
                label_id += 1







TempateExpression{body=[Mid-level-ontology.Tunnel], head=[?]}
TempateExpression{body=[?], head=[SUMO.StationaryArtifact]}
TempateExpression{body=[transport.RailwayJunction], head=[?]}
TempateExpression{body=[?], head=[Mid-level-ontology.TransitwayJunction]}
TempateExpression{body=[transport.MerchantMarine], head=[?]}
TempateExpression{body=[?], head=[SUMO.Collection]}
TempateExpression{body=[transport.NavigationLight], head=[?]}
TempateExpression{body=[?], head=[transport.AidToNavigation]}
TempateExpression{body=[transport.Barge], head=[?]}
TempateExpression{body=[?], head=[Mid-level-ontology.Watercraft]}
TempateExpression{body=[Mid-level-ontology.Bridge], head=[?]}
TempateExpression{body=[?], head=[SUMO.LandTransitway]}
TempateExpression{body=[Mid-level-ontology.Railway], head=[?]}
TempateExpression{body=[?], head=[SUMO.LandTransitway]}
TempateExpression{body=[Mid-level-ontology.Waterway], head=[?]}
TempateExpression{body=[?], head=[SUMO.WaterArea]}
TempateExpression{body=[transport.L

In [63]:
label_dict

{'TempateExpression{body=[Mid-level-ontology.Tunnel], head=[?]}': 0,
 'TempateExpression{body=[?], head=[SUMO.StationaryArtifact]}': 1,
 'TempateExpression{body=[transport.RailwayJunction], head=[?]}': 2,
 'TempateExpression{body=[?], head=[Mid-level-ontology.TransitwayJunction]}': 3,
 'TempateExpression{body=[transport.MerchantMarine], head=[?]}': 4,
 'TempateExpression{body=[?], head=[SUMO.Collection]}': 5,
 'TempateExpression{body=[transport.NavigationLight], head=[?]}': 6,
 'TempateExpression{body=[?], head=[transport.AidToNavigation]}': 7,
 'TempateExpression{body=[transport.Barge], head=[?]}': 8,
 'TempateExpression{body=[?], head=[Mid-level-ontology.Watercraft]}': 9,
 'TempateExpression{body=[Mid-level-ontology.Bridge], head=[?]}': 10,
 'TempateExpression{body=[?], head=[SUMO.LandTransitway]}': 11,
 'TempateExpression{body=[Mid-level-ontology.Railway], head=[?]}': 12,
 'TempateExpression{body=[Mid-level-ontology.Waterway], head=[?]}': 13,
 'TempateExpression{body=[?], head=[SUMO

In [69]:

# Save to cache
if label_dict_file:
    if verbose:
        print(f"💾 Saving label_dict to cache: {label_dict_file}")
    label_id_str = ''
    for lab_id, lab in enumerate(sorted(label_dict.keys(), key=lambda x: label_dict[x])):
        label_id_str += str(lab_id) + '\t' + lab + '\n'
    with open(label_dict_file, 'w', encoding='utf-8') as f:
        f.write(label_id_str)

In [71]:
label_id_str

'0\tTempateExpression{body=[Mid-level-ontology.Tunnel], head=[?]}\n1\tTempateExpression{body=[?], head=[SUMO.StationaryArtifact]}\n2\tTempateExpression{body=[transport.RailwayJunction], head=[?]}\n3\tTempateExpression{body=[?], head=[Mid-level-ontology.TransitwayJunction]}\n4\tTempateExpression{body=[transport.MerchantMarine], head=[?]}\n5\tTempateExpression{body=[?], head=[SUMO.Collection]}\n6\tTempateExpression{body=[transport.NavigationLight], head=[?]}\n7\tTempateExpression{body=[?], head=[transport.AidToNavigation]}\n8\tTempateExpression{body=[transport.Barge], head=[?]}\n9\tTempateExpression{body=[?], head=[Mid-level-ontology.Watercraft]}\n10\tTempateExpression{body=[Mid-level-ontology.Bridge], head=[?]}\n11\tTempateExpression{body=[?], head=[SUMO.LandTransitway]}\n12\tTempateExpression{body=[Mid-level-ontology.Railway], head=[?]}\n13\tTempateExpression{body=[Mid-level-ontology.Waterway], head=[?]}\n14\tTempateExpression{body=[?], head=[SUMO.WaterArea]}\n15\tTempateExpression{bod

In [72]:
num_labels = len(label_dict)
if verbose:
    print(f"📊 Number of labels: {num_labels}")

📊 Number of labels: 379


In [73]:
label_dict


{'TempateExpression{body=[Mid-level-ontology.Tunnel], head=[?]}': 0,
 'TempateExpression{body=[?], head=[SUMO.StationaryArtifact]}': 1,
 'TempateExpression{body=[transport.RailwayJunction], head=[?]}': 2,
 'TempateExpression{body=[?], head=[Mid-level-ontology.TransitwayJunction]}': 3,
 'TempateExpression{body=[transport.MerchantMarine], head=[?]}': 4,
 'TempateExpression{body=[?], head=[SUMO.Collection]}': 5,
 'TempateExpression{body=[transport.NavigationLight], head=[?]}': 6,
 'TempateExpression{body=[?], head=[transport.AidToNavigation]}': 7,
 'TempateExpression{body=[transport.Barge], head=[?]}': 8,
 'TempateExpression{body=[?], head=[Mid-level-ontology.Watercraft]}': 9,
 'TempateExpression{body=[Mid-level-ontology.Bridge], head=[?]}': 10,
 'TempateExpression{body=[?], head=[SUMO.LandTransitway]}': 11,
 'TempateExpression{body=[Mid-level-ontology.Railway], head=[?]}': 12,
 'TempateExpression{body=[Mid-level-ontology.Waterway], head=[?]}': 13,
 'TempateExpression{body=[?], head=[SUMO

In [74]:
if verbose:
    print("📝 Creating label matrix...")
labels = sp.lil_matrix((num_nodes, num_labels))

📝 Creating label matrix...
